# 04 — Performance senza autoinganni
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

Il notebook esplora correttezza e ambiente, NON sostituisce BenchmarkDotNet.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using System.Runtime.InteropServices;
Console.WriteLine(RuntimeInformation.FrameworkDescription);
Console.WriteLine($"SIMD={Vector.IsHardwareAccelerated}, double lanes={Vector<double>.Count}");
var data = Enumerable.Range(0,4099).Select(i => (i%17)*0.125).ToArray();
Console.WriteLine(NumericAlgorithms.Dot<double>(data,data));
Console.WriteLine(Signals.VectorDot(data,data));

## Da terminale, nella root del repository
```sh
dotnet run -c Release --project src/benchmarks/MathemaSharp.Benchmarks -- --filter '*SumBenchmarks*' --job Short --artifacts ops/artifacts/benchmarks
dotnet run -c Release --project src/benchmarks/MathemaSharp.Benchmarks -- --filter '*DotBenchmarks*' --job Short --artifacts ops/artifacts/benchmarks
```
Benchmark: somma manuale/generica/LINQ/Kahan, float/double/decimal, dot scalare/SIMD, BigInteger factorial, DFT Complex. Input preparati in GlobalSetup, risultati restituiti, MemoryDiagnoser attivo.
Confrontare Mean, Error, Ratio e Allocated per stessa N; annotare CPU, runtime, JIT e GC dal report. `--job Dry` è solo smoke test: un campione cold-start non consente conclusioni. Il kernel del notebook è un processo Roslyn scripting, non un ambiente di misura. Generic Math non implica SIMD; SIMD riordina la riduzione.